In [1]:
import cyvcf2
import pandas as pd

# ── Hardcoded inputs ───────────────────────────────────────────────────────────
SAMPLE    = "SBC10"
VCF_PATH  = "../results/sift4g/SBC10.private.sift4g.vcf.gz"
DMR_PATH  = "../results/DMR/DMR_annotated.tsv"
GENE_INFO = "../resources/NCBI_FTP/gene_info_4558"

# ── Constants ──────────────────────────────────────────────────────────────────
IMPACT_ORDER = ["HIGH", "MODERATE", "LOW", "MODIFIER"]
IMPACT_SCORE = {"HIGH": 3, "MODERATE": 2, "LOW": 1, "MODIFIER": 0}

# Multi-omics Gene Ranking — Scoring Review

Notebook counterpart of `workflow/scripts/build_ranked_genes.py`.
Produces a `df` DataFrame with all intermediate and final scores for interactive exploration.

In [2]:
def load_sorbi_to_loc(gene_info_path: str) -> dict:
    gi = pd.read_csv(gene_info_path, sep="\t", usecols=["Symbol", "LocusTag"])
    return gi.set_index("LocusTag")["Symbol"].to_dict()


def parse_vcf(vcf_path: str, sorbi_to_loc: dict):
    snpeff_hits: dict[str, list] = {}
    sift_hits:   dict[str, list] = {}

    vcf = cyvcf2.VCF(vcf_path)
    for variant in vcf:
        ann = variant.INFO.get("ANN")
        if ann:
            for entry in str(ann).split(","):
                parts = entry.split("|")
                if len(parts) < 4:
                    continue
                impact   = parts[2]
                sorbi_id = parts[3]
                if impact not in IMPACT_SCORE or not sorbi_id.startswith("SORBI_"):
                    continue
                loc_id = sorbi_to_loc.get(sorbi_id)
                if loc_id:
                    snpeff_hits.setdefault(loc_id, []).append(impact)

        siftinfo = variant.INFO.get("SIFTINFO")
        if siftinfo:
            for entry in str(siftinfo).split(","):
                parts = entry.split("|")
                if len(parts) < 13:
                    continue
                loc_id    = parts[2]
                sift_pred = parts[12]
                score_str = parts[8]
                if sift_pred not in ("TOLERATED", "DAMAGING"):
                    continue
                try:
                    sift_hits.setdefault(loc_id, []).append(float(score_str))
                except ValueError:
                    pass

    return snpeff_hits, sift_hits


def build_genomics(snpeff_hits: dict, sift_hits: dict) -> pd.DataFrame:
    all_genes = set(snpeff_hits) | set(sift_hits)
    rows = []
    for gene in all_genes:
        impacts = snpeff_hits.get(gene, [])
        scores  = sift_hits.get(gene, [])

        worst_impact = (
            min(impacts, key=lambda x: IMPACT_ORDER.index(x))
            if impacts else "MODIFIER"
        )
        impact_score = IMPACT_SCORE[worst_impact]

        if scores:
            min_sift        = min(scores)
            sift_disruption = 1.0 - min_sift
        else:
            min_sift        = None
            sift_disruption = 0.0

        rows.append({
            "gene_label":      gene,
            "worst_impact":    worst_impact,
            "n_variants":      len(impacts),
            "impact_score":    round(impact_score, 4),
            "n_sift_scored":   len(scores),
            "min_sift_score":  round(min_sift, 4) if min_sift is not None else None,
            "sift_disruption": round(sift_disruption, 4),
            "genomic_score":   round(impact_score + sift_disruption, 4),
        })

    return pd.DataFrame(rows)


def build_epigenomics(dmr_path: str, sample: str) -> pd.DataFrame:
    dmr = pd.read_csv(dmr_path, sep="\t")
    dmr = dmr[(dmr["sample_a"] == sample) | (dmr["sample_b"] == sample)].copy()
    dmr = dmr.dropna(subset=["gene_label"])
    dmr = dmr[dmr["gene_label"].astype(str).str.strip() != ""]
    dmr["abs_methy"] = dmr["diff.Methy"].abs()

    idx_max = dmr.groupby("gene_label")["abs_methy"].idxmax()
    epi = dmr.loc[idx_max, ["gene_label", "abs_methy", "direction", "feature"]].copy()
    epi = epi.rename(columns={"abs_methy": "epigenomic_score"})
    epi["epigenomic_score"] = epi["epigenomic_score"].round(4)
    return epi.reset_index(drop=True)

## Epigenomics, DMR annotation quick peek

In [7]:
# for genomics score, the data is obtained from the annotated VCF file (ann by SnpEff and SIFT4G)
# for epigenomics, the sole input is DMR annotate tsv file
ann_dmr_df = pd.read_csv(DMR_PATH, sep="\t")
ann_dmr_df

,chr,start,end,length,nCG,meanMethy1,meanMethy2,diff.Methy,areaStat,sample_a,sample_b,direction,feature,gene_label
0,NC_012873.2,67594394,67601630,7237,3447,0.244203,0.056058,0.188145,19719.767762,SBC10,SBC4,hyper_SBC10,promoter,LOC110435067
1,NC_012870.2,72402810,72407599,4790,2449,0.218997,0.045744,0.173253,15737.519672,SBC10,SBC4,hyper_SBC10,intergenic,NaN
2,NC_012870.2,8087755,8092275,4521,2413,0.265127,0.096703,0.168424,13173.727692,SBC10,SBC4,hyper_SBC10,intergenic,NaN
3,NC_012873.2,10883151,10887512,4362,2303,0.256243,0.080646,0.175597,13112.356467,SBC10,SBC4,hyper_SBC10,promoter,LOC8078332
4,NC_012870.2,46238158,46242655,4498,2064,0.225253,0.052197,0.173056,13096.312671,SBC10,SBC4,hyper_SBC10,intergenic,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77067,NC_012871.2,64440351,64442061,1711,41,0.282175,0.292458,-0.010283,6.733625,SBC10,SBC23,hyper_SBC23,promoter,LOC8054790
77068,NC_012871.2,64440351,64442061,1711,41,0.282175,0.292458,-0.010283,6.733625,SBC10,SBC23,hyper_SBC23,exon,LOC8054790
77069,NC_012871.2,64440351,64442061,1711,41,0.282175,0.292458,-0.010283,6.733625,SBC10,SBC23,hyper_SBC23,CDS,LOC8054790
77070,NC_012879.2,4313138,4314832,1695,80,0.245333,0.174466,0.070866,-2.514123,SBC10,SBC23,hyper_SBC10,promoter,LOC8072571


In [12]:
dmr_sbc10_promoter = ann_dmr_df[
    ((ann_dmr_df["sample_a"] == SAMPLE) | (ann_dmr_df["sample_b"] == SAMPLE))
    & ann_dmr_df["gene_label"].notna()
    & (ann_dmr_df["gene_label"].str.strip() != "")
    & (ann_dmr_df["feature"] == "promoter")
].drop_duplicates(subset=["chr", "start", "end", "sample_a", "sample_b", "gene_label"])

gene_directions = (
    dmr_sbc10_promoter.groupby("gene_label")["direction"]
    .agg(set)
    .rename("directions")
    .reset_index()
)

conflicting = gene_directions[
    gene_directions["directions"].apply(
        lambda d: "hyper_SBC10" in d and len(d) > 1
    )
].copy()

conflicting["directions"] = conflicting["directions"].apply(lambda d: ", ".join(sorted(d)))
print(f"Genes with conflicting promoter DMR direction (hyper_SBC10 in ≥1 pair): {len(conflicting)}")
conflicting

Genes with conflicting promoter DMR direction (hyper_SBC10 in ≥1 pair): 479


,gene_label,directions
18,LOC110429618,"hyper_SBC10, hyper_SBC4"
30,LOC110429668,"hyper_SBC10, hyper_SBC11, hyper_SBC4"
45,LOC110429730,"hyper_SBC10, hyper_SBC11"
58,LOC110429787,"hyper_SBC10, hyper_SBC11"
61,LOC110429796,"hyper_SBC10, hyper_SBC4"
...,...,...
7083,TRNAM-CAU,"hyper_SBC10, hyper_SBC23"
7089,TRNAR-ACG,"hyper_SBC10, hyper_SBC4"
7091,TRNAR-UCU,"hyper_SBC10, hyper_SBC23"
7096,TRNAV-AAC,"hyper_SBC10, hyper_SBC23"


## Checking the scoring method's temporary outcome

In [3]:
sorbi_to_loc = load_sorbi_to_loc(GENE_INFO)

snpeff_hits, sift_hits = parse_vcf(VCF_PATH, sorbi_to_loc)
print(f"SnpEff-mapped genes : {len(snpeff_hits):,}")
print(f"SIFT-scored genes   : {len(sift_hits):,}")

genomics    = build_genomics(snpeff_hits, sift_hits)
epigenomics = build_epigenomics(DMR_PATH, SAMPLE)
print(f"Genomics table      : {len(genomics):,} genes")
print(f"Epigenomics table   : {len(epigenomics):,} genes")

df = genomics.merge(epigenomics, on="gene_label", how="outer")
df["genomic_score"]    = df["genomic_score"].fillna(0.0)
df["epigenomic_score"] = df["epigenomic_score"].fillna(0.0)
df["worst_impact"]     = df["worst_impact"].fillna("MODIFIER")
df["impact_score"]     = df["impact_score"].fillna(0.0)
df["sift_disruption"]  = df["sift_disruption"].fillna(0.0)
df["n_variants"]       = df["n_variants"].fillna(0).astype(int)
df["n_sift_scored"]    = df["n_sift_scored"].fillna(0).astype(int)

max_g = df["genomic_score"].max()
max_e = df["epigenomic_score"].max()
df["variant_score"]     = (df["genomic_score"]    / max_g).round(4)
df["methylation_score"] = (df["epigenomic_score"] / max_e).round(4)

df = df.sort_values(["variant_score", "methylation_score"], ascending=False).reset_index(drop=True)

both = (df["genomic_score"] > 0) & (df["epigenomic_score"] > 0)
print(f"Genes in both layers: {both.sum():,}  /  total: {len(df):,}")

df

SnpEff-mapped genes : 16,815
SIFT-scored genes   : 2,563
Genomics table      : 17,233 genes
Epigenomics table   : 8,381 genes
Genes in both layers: 932  /  total: 20,812


,gene_label,worst_impact,n_variants,impact_score,n_sift_scored,min_sift_score,sift_disruption,genomic_score,epigenomic_score,direction,feature,variant_score,methylation_score
0,LOC8075699,HIGH,27,3.0,4,0.05,0.95,3.95,0.3201,hyper_SBC23,promoter,1.0,0.4275
1,LOC8066704,HIGH,377,3.0,89,0.05,0.95,3.95,0.2383,hyper_SBC4,promoter,1.0,0.3182
2,LOC8081127,HIGH,102,3.0,49,0.05,0.95,3.95,0.2316,hyper_SBC10,promoter,1.0,0.3093
3,LOC8067156,HIGH,63,3.0,27,0.05,0.95,3.95,0.1687,hyper_SBC10,promoter,1.0,0.2253
4,LOC8061352,HIGH,171,3.0,11,0.05,0.95,3.95,0.1482,hyper_SBC10,promoter,1.0,0.1979
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20807,LOC8155667,MODIFIER,14,0.0,0,NaN,0.00,0.00,0.0000,NaN,NaN,0.0,0.0000
20808,LOC8155679,MODIFIER,4,0.0,0,NaN,0.00,0.00,0.0000,NaN,NaN,0.0,0.0000
20809,LOC8155680,MODIFIER,11,0.0,0,NaN,0.00,0.00,0.0000,NaN,NaN,0.0,0.0000
20810,LOC8155683,MODIFIER,8,0.0,0,NaN,0.00,0.00,0.0000,NaN,NaN,0.0,0.0000


In [4]:
# Genes with evidence in both omics layers
df_both = df[(df["genomic_score"] > 0) & (df["epigenomic_score"] > 0)].copy()
print(f"Both-layer genes: {len(df_both)}")
df_both.head(20)

Both-layer genes: 932


,gene_label,worst_impact,n_variants,impact_score,n_sift_scored,min_sift_score,sift_disruption,genomic_score,epigenomic_score,direction,feature,variant_score,methylation_score
0,LOC8075699,HIGH,27,3.0,4,0.05,0.95,3.95,0.3201,hyper_SBC23,promoter,1.0000,0.4275
1,LOC8066704,HIGH,377,3.0,89,0.05,0.95,3.95,0.2383,hyper_SBC4,promoter,1.0000,0.3182
2,LOC8081127,HIGH,102,3.0,49,0.05,0.95,3.95,0.2316,hyper_SBC10,promoter,1.0000,0.3093
3,LOC8067156,HIGH,63,3.0,27,0.05,0.95,3.95,0.1687,hyper_SBC10,promoter,1.0000,0.2253
4,LOC8061352,HIGH,171,3.0,11,0.05,0.95,3.95,0.1482,hyper_SBC10,promoter,1.0000,0.1979
5,LOC110430745,HIGH,184,3.0,3,0.05,0.95,3.95,0.1387,hyper_SBC4,promoter,1.0000,0.1852
6,LOC8074840,HIGH,156,3.0,84,0.05,0.95,3.95,0.1306,hyper_SBC10,exon,1.0000,0.1744
26,LOC8070962,HIGH,546,3.0,123,0.06,0.94,3.94,0.2637,hyper_SBC23,exon,0.9975,0.3522
27,LOC8071161,HIGH,90,3.0,5,0.06,0.94,3.94,0.2580,hyper_SBC23,exon,0.9975,0.3446
28,LOC8062208,HIGH,48,3.0,39,0.06,0.94,3.94,0.1638,hyper_SBC10,promoter,0.9975,0.2188


In [5]:
# Score distributions
df[["genomic_score", "epigenomic_score", "variant_score", "methylation_score"]].describe()

,genomic_score,epigenomic_score,variant_score,methylation_score
count,20812.000000,20812.000000,20812.000000,20812.000000
mean,0.330886,0.070275,0.083769,0.093850
std,0.835674,0.095246,0.211562,0.127198
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.143200,0.000000,0.191200
max,3.950000,0.748800,1.000000,1.000000


---
## Raw data (pre-aggregation)

`df_variants` — one row per variant site, before collapsing to gene level.  
`df_dmr` — one row per unique DMR region, before taking the per-gene maximum.

In [ ]:
def parse_vcf_raw(vcf_path: str, sorbi_to_loc: dict) -> pd.DataFrame:
    """One row per variant site. Picks the worst SnpEff impact and minimum SIFT score
    across all annotations for that site."""
    rows = []
    vcf = cyvcf2.VCF(vcf_path)
    for variant in vcf:
        best_impact = None
        best_effect = None
        best_gene   = None

        ann = variant.INFO.get("ANN")
        if ann:
            for entry in str(ann).split(","):
                parts = entry.split("|")
                if len(parts) < 4:
                    continue
                impact   = parts[2]
                effect   = parts[1]
                sorbi_id = parts[3]
                if impact not in IMPACT_SCORE or not sorbi_id.startswith("SORBI_"):
                    continue
                if best_impact is None or IMPACT_ORDER.index(impact) < IMPACT_ORDER.index(best_impact):
                    best_impact = impact
                    best_effect = effect
                    best_gene   = sorbi_to_loc.get(sorbi_id)

        min_sift  = None
        sift_pred = None
        siftinfo  = variant.INFO.get("SIFTINFO")
        if siftinfo:
            for entry in str(siftinfo).split(","):
                parts = entry.split("|")
                if len(parts) < 13:
                    continue
                pred      = parts[12]
                score_str = parts[8]
                if pred not in ("TOLERATED", "DAMAGING"):
                    continue
                try:
                    score = float(score_str)
                    if min_sift is None or score < min_sift:
                        min_sift  = score
                        sift_pred = pred
                except ValueError:
                    pass

        rows.append({
            "chrom":           variant.CHROM,
            "pos":             variant.POS,
            "ref":             variant.REF,
            "alt":             ",".join(variant.ALT),
            "impact":          best_impact,
            "effect":          best_effect,
            "gene_label":      best_gene,
            "sift_score":      round(min_sift, 4) if min_sift is not None else None,
            "sift_prediction": sift_pred,
        })

    return pd.DataFrame(rows)


df_variants = parse_vcf_raw(VCF_PATH, sorbi_to_loc)
print(f"Total variant sites : {len(df_variants):,}")
print(df_variants["impact"].value_counts())
df_variants.head(20)

In [ ]:
# Raw DMR table — deduplicated to one row per unique DMR region.
# ann_dmr_df has multiple rows per DMR when it overlaps multiple features/genes;
# drop_duplicates on coordinates + pair collapses back to the DMR level.
df_dmr = (
    ann_dmr_df[
        (ann_dmr_df["sample_a"] == SAMPLE) | (ann_dmr_df["sample_b"] == SAMPLE)
    ]
    .drop_duplicates(subset=["chr", "start", "end", "sample_a", "sample_b"])
    .reset_index(drop=True)
)
print(f"Unique DMR sites involving {SAMPLE}: {len(df_dmr):,}")
print(df_dmr["direction"].value_counts())
df_dmr.head(20)